# MK1 RSI Colab 学習・API

ColabでDrive上のcheckpointを読み込み、ローカルRSIクライアントから受け取った確認済み10次元特徴量でHead Bを追加学習します。これは報酬・方策を使う強化学習ではなく、Head Bのオンライン適合学習です。生パケットや通信内容は受け付けません。

事前にColabランタイムへWireGuardのフルトンネルを設定してください。Drive mountより前にIPv4/IPv6の全経路がWireGuardを使うことを検証し、不成立なら停止します。MK1Ai管理APIの要求/応答/モデルはML-KEM-768とAES-256-GCMでも暗号化します。WireGuard標準暗号はCurve25519/ChaCha20-Poly1305です。Google管理のDrive API/TLSは独立しており、暗号方式をこちらから固定できません。

学習レコードは `label=0` が確認済み正常、`label=1` が確認済み脅威です。両方のラベルが含まれない要求は拒否します。Head Bは正常度を出力するため、学習ターゲットは正常で1・脅威で0へ変換します。自動収集された正常候補だけをRSIへ送っても学習しません。誤検知/脅威の判定を人が確認し、`ai_data/curated/reviewed_feedback.jsonl` に記録してください。

上から順に実行してください。Flaskがない場合は自動で導入します。公開APIはBearer token必須、loopback bind、JSONサイズ制限付きです。Tunnel URLとtokenはランタイムごとに変わります。URLとtokenをローカル側へ設定したら、この出力を含むnotebookを共有・公開しないでください。APIは空いているloopback portを自動選択するため、固定portの競合を避けます。

最後のセルはTunnel URLのhealth応答と認証を自動検証します。想定APIであることが確認できない場合は、URLとtokenを表示せず停止します。APIが202を返すのは学習キューへの受付完了です。Colabの学習ログで `Fine-tuned Head B ...` と `Saved SHA-256:` が表示されて初めて、学習とDrive保存が完了します。

In [ ]:
import base64
import hashlib
import hmac
import importlib.util
import json
import math
import os
import queue
import secrets
import shutil
import socket
import subprocess
import sys
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

def require_wireguard_full_tunnel():
    interface = os.environ.get('MK1_WIREGUARD_INTERFACE', 'wg0').strip()
    if not interface or len(interface) > 15 or not all(c.isalnum() or c in '_.-' for c in interface):
        raise RuntimeError('Set a valid MK1_WIREGUARD_INTERFACE before running this notebook.')
    try:
        allowed = subprocess.run(['wg', 'show', interface, 'allowed-ips'], check=True, capture_output=True, text=True, timeout=3).stdout
        route4 = subprocess.run(['ip', 'route', 'get', '1.1.1.1'], check=True, capture_output=True, text=True, timeout=3).stdout
        route6 = subprocess.run(['ip', '-6', 'route', 'get', '2606:4700:4700::1111'], check=True, capture_output=True, text=True, timeout=3).stdout
    except (OSError, subprocess.SubprocessError) as exc:
        raise RuntimeError('WireGuard full-tunnel verification failed; no Google/Colab access is allowed.') from exc
    if '0.0.0.0/0' not in allowed or '::/0' not in allowed or f'dev {interface}' not in route4 or f'dev {interface}' not in route6:
        raise RuntimeError('WireGuard must carry both IPv4 and IPv6 default routes before using Google/Colab.')
    return interface

WIREGUARD_INTERFACE = require_wireguard_full_tunnel()
try:
    from pqcrypto.kem.ml_kem_768 import decaps, encaps, keygen
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pqcrypto==1.0.0'])
    from pqcrypto.kem.ml_kem_768 import decaps, encaps, keygen
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

ML_KEM_PUBLIC_KEY, ML_KEM_PRIVATE_KEY = keygen()
ML_KEM_PUBLIC_KEY_SHA256 = hashlib.sha256(ML_KEM_PUBLIC_KEY).hexdigest()

def derive_transport_key(shared_secret, aad):
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=hashlib.sha256(aad).digest(), info=b'MK1-RSI-ML-KEM-768-AES-256-GCM-v1').derive(shared_secret)

def transport_aad(method, path, fingerprint, kem_ciphertext):
    return b'MK1-RSI-v1\0' + b'\0'.join((method.upper().encode('ascii'), path.encode('utf-8'), fingerprint.encode('ascii'), kem_ciphertext))

def decrypt_api_request():
    kem_ciphertext = base64.b64decode(request.headers.get('X-MK1-KEM', ''), validate=True)
    aad = transport_aad(request.method, request.path, ML_KEM_PUBLIC_KEY_SHA256, kem_ciphertext)
    key = derive_transport_key(decaps(ML_KEM_PRIVATE_KEY, kem_ciphertext), aad)
    encrypted_body = request.get_data(cache=False)
    if len(encrypted_body) < 28:
        raise ValueError('encrypted request is truncated')
    document = json.loads(AESGCM(key).decrypt(encrypted_body[:12], encrypted_body[12:], aad).decode('utf-8'))
    if not isinstance(document, dict) or set(document) != {'token', 'payload'}:
        raise ValueError('invalid encrypted request schema')
    return document, key, aad

def encrypted_api_response(value, key, aad, status=200):
    plaintext = value if isinstance(value, bytes) else json.dumps(value, separators=(',', ':'), allow_nan=False).encode('utf-8')
    nonce = secrets.token_bytes(12)
    return Response(nonce + AESGCM(key).encrypt(nonce, plaintext, aad + b'\0response'), status=status, mimetype='application/octet-stream')

from google.colab import drive
drive.mount('/content/drive')

if importlib.util.find_spec('flask') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'flask'])

import torch
from torch import nn
from flask import Flask, Response, jsonify, request

DRIVE_DIR = Path('/content/drive/MyDrive/MK1-Ai')
MODEL_PATH = DRIVE_DIR / 'cloud_base_model.pth'
HASH_PATH = DRIVE_DIR / 'cloud_base_model.pth.sha256'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
API_TOKEN = secrets.token_urlsafe(32)
print('Training device:', DEVICE)
if DEVICE.type != 'cuda':
    print('GPU runtime is unavailable; training will run on CPU. Select a GPU runtime for faster training.')

In [ ]:
class LightweightMultiTaskAI(nn.Module):
    def __init__(self, input_dim=10):
        super().__init__()
        self.shared_layer = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
        )
        self.head_a = nn.Sequential(
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        )
        self.head_b = nn.Sequential(nn.Linear(16, 1), nn.Sigmoid())
        self.head_c = nn.Sequential(nn.Linear(16, 1), nn.Sigmoid())

    def forward(self, values):
        shared = self.shared_layer(values)
        return self.head_a(shared), self.head_b(shared), self.head_c(shared)

def unwrap_state_dict(value):
    if isinstance(value, dict) and 'state_dict' in value:
        return value['state_dict']
    if isinstance(value, dict) and 'model_state_dict' in value:
        return value['model_state_dict']
    return value

def checked_state_dict(value):
    candidate = LightweightMultiTaskAI()
    expected = candidate.state_dict()
    source = unwrap_state_dict(value)
    if not isinstance(source, dict):
        raise ValueError('state_dict must be a dictionary')

    if set(source) == set(expected):
        checked = {}
        for key, reference in expected.items():
            tensor = torch.as_tensor(source[key], dtype=reference.dtype, device='cpu')
            if tuple(tensor.shape) != tuple(reference.shape) or not torch.isfinite(tensor).all():
                raise ValueError(f'invalid tensor shape or value: {key}')
            checked[key] = tensor
        candidate.load_state_dict(checked, strict=True)
        return checked

    legacy_shapes = {
        'shared_layer.0.weight': (32, 10),
        'shared_layer.0.bias': (32,),
        'shared_layer.2.weight': (16, 32),
        'shared_layer.2.bias': (16,),
        'head_a.0.weight': (1, 16),
        'head_a.0.bias': (1,),
        'head_b.0.weight': (1, 16),
        'head_b.0.bias': (1,),
        'head_c.0.weight': (1, 16),
        'head_c.0.bias': (1,),
    }
    if set(source) != set(legacy_shapes):
        raise ValueError('state_dict keys do not match current or supported legacy architecture')

    legacy = {}
    for key, shape in legacy_shapes.items():
        tensor = torch.as_tensor(source[key], device='cpu')
        if tuple(tensor.shape) != shape or not torch.isfinite(tensor).all():
            raise ValueError(f'invalid legacy tensor shape or value: {key}')
        legacy[key] = tensor

    checked = {key: torch.zeros_like(reference, device='cpu') for key, reference in expected.items()}
    checked['shared_layer.0.weight'][:32].copy_(legacy['shared_layer.0.weight'])
    checked['shared_layer.0.bias'][:32].copy_(legacy['shared_layer.0.bias'])
    checked['shared_layer.2.weight'][:16, :32].copy_(legacy['shared_layer.2.weight'])
    checked['shared_layer.2.bias'][:16].copy_(legacy['shared_layer.2.bias'])
    checked['shared_layer.4.weight'][:16, :16].copy_(torch.eye(16))
    checked['head_a.0.weight'][0].copy_(legacy['head_a.0.weight'][0])
    checked['head_a.0.weight'][1].copy_(-legacy['head_a.0.weight'][0])
    checked['head_a.0.bias'][0].copy_(legacy['head_a.0.bias'][0])
    checked['head_a.0.bias'][1].copy_(-legacy['head_a.0.bias'][0])
    checked['head_a.2.weight'][0, :2] = torch.tensor([1.0, -1.0])
    for key in ('head_b.0.weight', 'head_b.0.bias', 'head_c.0.weight', 'head_c.0.bias'):
        checked[key].copy_(legacy[key])
    candidate.load_state_dict(checked, strict=True)
    print('Migrated supported legacy checkpoint to the current architecture.')
    return checked

model_lock = threading.RLock()
model = LightweightMultiTaskAI()
if MODEL_PATH.is_file():
    saved = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
    model.load_state_dict(checked_state_dict(saved), strict=True)
    print('Loaded Drive checkpoint:', MODEL_PATH)
else:
    print('No Drive checkpoint yet; first accepted local state initializes the model.')
model.to(DEVICE).eval()

In [ ]:
def clean_records(records):
    cleaned = []
    if not isinstance(records, list):
        return cleaned
    for record in records[:2048]:
        if not isinstance(record, dict):
            continue
        features = record.get('features')
        label = record.get('label')
        if not isinstance(features, list) or len(features) != 10:
            continue
        if isinstance(label, bool) or label not in (0, 1):
            continue
        if any(isinstance(item, bool) or not isinstance(item, (int, float)) for item in features):
            continue
        values = [float(item) for item in features]
        if all(math.isfinite(item) and 0.0 <= item <= 1.0 for item in values):
            cleaned.append({'features': values, 'label': int(label)})
    return cleaned

def atomic_save_checkpoint():
    fd, temporary_name = tempfile.mkstemp(prefix='.cloud-model-', suffix='.tmp', dir=DRIVE_DIR)
    os.close(fd)
    temporary_path = Path(temporary_name)
    hash_temporary = None
    try:
        with model_lock:
            checkpoint = {
                'format_version': 2,
                'state_dict': {key: value.detach().cpu() for key, value in model.state_dict().items()},
                'reviewed_labels': [0, 1],
            }
            torch.save(checkpoint, temporary_path)
        with temporary_path.open('rb') as handle:
            os.fsync(handle.fileno())
            digest = hashlib.sha256()
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(chunk)
        hash_value = digest.hexdigest()
        hash_fd, hash_name = tempfile.mkstemp(prefix='.cloud-hash-', suffix='.tmp', dir=DRIVE_DIR)
        hash_temporary = Path(hash_name)
        with os.fdopen(hash_fd, 'w', encoding='ascii') as handle:
            handle.write(f'{hash_value}  cloud_base_model.pth\n')
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary_path, MODEL_PATH)
        os.replace(hash_temporary, HASH_PATH)
        return hash_value
    finally:
        temporary_path.unlink(missing_ok=True)
        if hash_temporary is not None:
            hash_temporary.unlink(missing_ok=True)

training_queue = queue.Queue(maxsize=2)
def train_worker():
    while True:
        records = training_queue.get()
        try:
            if records is None:
                return
            labels = {record['label'] for record in records}
            if labels != {0, 1}:
                print('Training request skipped: reviewed benign and threat labels are both required.')
                continue
            with model_lock:
                model.train()
                for name, parameter in model.named_parameters():
                    parameter.requires_grad = name.startswith('head_b')
                optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-3)
                loss_fn = nn.BCELoss(reduction='none')
                features = [record['features'] for record in records]
                targets = torch.tensor([[1.0 - record['label']] for record in records], dtype=torch.float32, device=DEVICE)
                sample_weights = torch.where(
                    targets > 0.5,
                    len(records) / (2.0 * (targets > 0.5).sum()),
                    len(records) / (2.0 * (targets <= 0.5).sum()),
                )
                for start in range(0, len(features), 100):
                    batch = torch.tensor(features[start:start + 100], dtype=torch.float32, device=DEVICE)
                    batch_targets = targets[start:start + 100]
                    batch_weights = sample_weights[start:start + 100]
                    optimizer.zero_grad(set_to_none=True)
                    _, prediction, _ = model(batch)
                    loss = (loss_fn(prediction, batch_targets) * batch_weights).mean()
                    loss.backward()
                    optimizer.step()
                model.eval()
            print(f'Fine-tuned Head B on {len(features)} reviewed rows; loss={float(loss.detach()):.5f}')
            print('Saved SHA-256:', atomic_save_checkpoint())
        except Exception as exc:
            print('Training request failed:', type(exc).__name__, str(exc))
        finally:
            training_queue.task_done()

threading.Thread(target=train_worker, daemon=True, name='gpu-finetune').start()

In [ ]:
app = Flask('mk1_colab_model_api')
app.config['MAX_CONTENT_LENGTH'] = 1024 * 1024

@app.errorhandler(413)
def request_too_large(_error):
    return jsonify({'error': 'request too large'}), 413

@app.get('/healthz')
def health_check():
    return jsonify({'status': 'ok', 'service': 'mk1-colab-rsi', 'auth_required': True, 'pqc': 'ML-KEM-768', 'aead': 'AES-256-GCM'})

@app.get('/crypto/public-key')
def public_key():
    return jsonify({'algorithm': 'ML-KEM-768', 'public_key': base64.b64encode(ML_KEM_PUBLIC_KEY).decode('ascii'), 'sha256': ML_KEM_PUBLIC_KEY_SHA256})

def authenticated_encrypted_request():
    try:
        document, key, aad = decrypt_api_request()
    except Exception:
        return None, None, None, (jsonify({'error': 'invalid encrypted request'}), 400)
    if not isinstance(document['token'], str) or not hmac.compare_digest(document['token'], API_TOKEN):
        return document, key, aad, encrypted_api_response({'error': 'unauthorized'}, key, aad, 401)
    return document, key, aad, None

@app.post('/rsi')
def receive_rsi():
    document, key, aad, denied = authenticated_encrypted_request()
    if denied is not None:
        return denied
    payload = document['payload']
    required_payload_keys = {'task', 'curated_data', 'training', 'submitted_at'}
    if not isinstance(payload, dict) or set(payload) != required_payload_keys:
        return encrypted_api_response({'error': 'invalid feature-only payload'}, key, aad, 400)
    if payload.get('task') != 'recursive_self_improvement':
        return encrypted_api_response({'error': 'unsupported task'}, key, aad, 400)
    curated = payload.get('curated_data')
    if not isinstance(curated, dict) or set(curated) != {'records', 'contents_uploaded'}:
        return encrypted_api_response({'error': 'invalid curated data'}, key, aad, 400)
    if not isinstance(curated['contents_uploaded'], bool):
        return encrypted_api_response({'error': 'invalid contents_uploaded flag'}, key, aad, 400)
    records = curated.get('records')
    if not isinstance(records, list) or not records:
        return encrypted_api_response({'error': 'no feature records'}, key, aad, 422)
    if len(records) > 2048:
        return encrypted_api_response({'error': 'too many feature records'}, key, aad, 413)
    cleaned_records = clean_records(records)
    if not cleaned_records:
        return encrypted_api_response({'error': 'no valid feature records'}, key, aad, 422)
    if {record['label'] for record in cleaned_records} != {0, 1}:
        return encrypted_api_response({'error': 'reviewed benign and threat labels are both required'}, key, aad, 422)
    try:
        training_queue.put_nowait(cleaned_records)
    except queue.Full:
        return encrypted_api_response({'error': 'training queue is full'}, key, aad, 503)
    return encrypted_api_response({'accepted': True, 'clean_rows': len(cleaned_records)}, key, aad, 202)

@app.post('/model.sha256')
def get_model_hash():
    document, key, aad, denied = authenticated_encrypted_request()
    if denied is not None:
        return denied
    if document['payload'] != {'action': 'model_hash'}:
        return encrypted_api_response({'error': 'invalid model hash request'}, key, aad, 400)
    if not HASH_PATH.is_file():
        return encrypted_api_response({'error': 'model not ready'}, key, aad, 404)
    return encrypted_api_response(HASH_PATH.read_bytes(), key, aad)

@app.post('/model')
def get_model():
    document, key, aad, denied = authenticated_encrypted_request()
    if denied is not None:
        return denied
    if document['payload'] != {'action': 'model'}:
        return encrypted_api_response({'error': 'invalid model request'}, key, aad, 400)
    if not MODEL_PATH.is_file():
        return encrypted_api_response({'error': 'model not ready'}, key, aad, 404)
    if MODEL_PATH.stat().st_size > 128 * 1024 * 1024:
        return encrypted_api_response({'error': 'model artifact exceeds the configured size limit'}, key, aad, 413)
    return encrypted_api_response(MODEL_PATH.read_bytes(), key, aad)

def make_test_request(payload, token=API_TOKEN):
    kem_ciphertext, shared_secret = encaps(ML_KEM_PUBLIC_KEY)
    aad = transport_aad('POST', '/rsi', ML_KEM_PUBLIC_KEY_SHA256, kem_ciphertext)
    key = derive_transport_key(shared_secret, aad)
    nonce = secrets.token_bytes(12)
    plaintext = json.dumps({'token': token, 'payload': payload}, separators=(',', ':')).encode('utf-8')
    body = nonce + AESGCM(key).encrypt(nonce, plaintext, aad)
    return body, {'X-MK1-KEM': base64.b64encode(kem_ciphertext).decode('ascii')}, key, aad

def decode_test_response(response, key, aad):
    body = response.get_data()
    plaintext = AESGCM(key).decrypt(body[:12], body[12:], aad + b'\0response')
    return json.loads(plaintext.decode('utf-8'))

valid_payload = {
    'task': 'recursive_self_improvement',
    'curated_data': {
        'records': [
            {'features': [0.1] * 10, 'label': 0},
            {'features': [0.9] * 10, 'label': 1},
        ],
        'contents_uploaded': True,
    },
    'training': {'epochs': 1, 'learning_rate': 0.001, 'batch_size': 100, 'device': 'cuda'},
    'submitted_at': int(time.time()),
}
queued_for_test = []
original_enqueue = training_queue.put_nowait
try:
    training_queue.put_nowait = lambda records: queued_for_test.append(records)
    with app.test_client() as client:
        assert client.get('/healthz').status_code == 200
        public_key = client.get('/crypto/public-key').get_json()
        assert public_key['algorithm'] == 'ML-KEM-768'
        assert public_key['sha256'] == ML_KEM_PUBLIC_KEY_SHA256
        assert client.post('/rsi', data=b'{}').status_code == 400
        body, headers, key, aad = make_test_request(valid_payload, token='invalid-token')
        denied = client.post('/rsi', data=body, headers=headers)
        assert denied.status_code == 401
        assert decode_test_response(denied, key, aad) == {'error': 'unauthorized'}
        one_class_payload = dict(valid_payload)
        one_class_payload['curated_data'] = {
            'records': [{'features': [0.1] * 10, 'label': 0}],
            'contents_uploaded': True,
        }
        body, headers, key, aad = make_test_request(one_class_payload)
        rejected = client.post('/rsi', data=body, headers=headers)
        assert rejected.status_code == 422
        assert 'both required' in decode_test_response(rejected, key, aad)['error']
        body, headers, key, aad = make_test_request(valid_payload)
        accepted = client.post('/rsi', data=body, headers=headers)
        assert accepted.status_code == 202
        assert decode_test_response(accepted, key, aad) == {'accepted': True, 'clean_rows': 2}
finally:
    training_queue.put_nowait = original_enqueue
assert queued_for_test == [[
    {'features': [0.1] * 10, 'label': 0},
    {'features': [0.9] * 10, 'label': 1},
]]
print('API smoke tests passed: auth, payload validation, both-class requirement, and accepted schema')

from werkzeug.serving import make_server
api_server = make_server('127.0.0.1', 0, app, threaded=True)
API_PORT = api_server.server_port
threading.Thread(target=api_server.serve_forever, daemon=True, name='model-api').start()
print(f'Authenticated API started on 127.0.0.1:{API_PORT}')

## Start the HTTPS tunnel

This downloads the Cloudflare Tunnel binary into the Colab VM and starts a temporary Quick Tunnel. Keep the notebook runtime alive. Copy both printed values to the local machine; configure `COLAB_RSI_TOKEN` as an environment variable or `model_sync_token` in `ai_data/config.json`.

In [ ]:
cloudflared_path = Path('/content/cloudflared')
if not cloudflared_path.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_path,
    )
    cloudflared_path.chmod(0o755)

if not os.access(cloudflared_path, os.X_OK):
    cloudflared_path.chmod(0o755)

tunnel = subprocess.Popen(
    [str(cloudflared_path), 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{API_PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()
def read_tunnel_output():
    if tunnel.stdout is None:
        return
    for line in tunnel.stdout:
        tunnel_lines.put(line.rstrip())

threading.Thread(target=read_tunnel_output, daemon=True, name='cloudflared-output').start()

public_url = None
deadline = time.monotonic() + 120
while time.monotonic() < deadline and tunnel.poll() is None:
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    if 'trycloudflare.com' in line:
        import re
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
if not public_url:
    tunnel.terminate()
    raise RuntimeError('Cloudflare URL was not ready. Restart the Colab runtime and rerun cells from the top.')

COLAB_RSI_ENDPOINT = public_url + '/rsi'
COLAB_MODEL_URL = public_url + '/model'
COLAB_MODEL_HASH_URL = public_url + '/model.sha256'
health_url = public_url + '/healthz'
verification_deadline = time.monotonic() + 45
last_verification_error = None
api_verified = False
while time.monotonic() < verification_deadline and tunnel.poll() is None:
    try:
        with urllib.request.urlopen(health_url, timeout=10) as response:
            health = json.loads(response.read().decode('utf-8'))
        if health.get('service') != 'mk1-colab-rsi' or health.get('auth_required') is not True or health.get('pqc') != 'ML-KEM-768' or health.get('aead') != 'AES-256-GCM':
            raise RuntimeError('Tunnel reached an unexpected service; credentials were not sent.')

        with urllib.request.urlopen(public_url + '/crypto/public-key', timeout=10) as response:
            public_key_document = json.loads(response.read().decode('utf-8'))
        remote_public_key = base64.b64decode(public_key_document['public_key'], validate=True)
        remote_fingerprint = hashlib.sha256(remote_public_key).hexdigest()
        if public_key_document.get('algorithm') != 'ML-KEM-768' or remote_fingerprint != public_key_document.get('sha256'):
            raise RuntimeError('Colab PQC key fingerprint validation failed.')
        kem_ciphertext, shared_secret = encaps(remote_public_key)
        aad = transport_aad('POST', '/rsi', remote_fingerprint, kem_ciphertext)
        test_key = derive_transport_key(shared_secret, aad)
        nonce = secrets.token_bytes(12)
        test_plaintext = json.dumps({'token': 'invalid-probe-token', 'payload': {}}, separators=(',', ':')).encode('utf-8')
        test_body = nonce + AESGCM(test_key).encrypt(nonce, test_plaintext, aad)
        unauth_request = urllib.request.Request(COLAB_RSI_ENDPOINT, data=test_body, headers={
            'Content-Type': 'application/octet-stream',
            'X-MK1-KEM': base64.b64encode(kem_ciphertext).decode('ascii'),
        }, method='POST')
        try:
            with urllib.request.urlopen(unauth_request, timeout=10) as response:
                unauth_status = response.status
                encrypted_error = response.read()
        except urllib.error.HTTPError as exc:
            unauth_status = exc.code
            encrypted_error = exc.read()
        if unauth_status != 401 or len(encrypted_error) < 28:
            raise RuntimeError(f'Encrypted authentication smoke test expected HTTP 401, got {unauth_status}.')
        decoded_error = AESGCM(test_key).decrypt(encrypted_error[:12], encrypted_error[12:], aad + b'\0response')
        if json.loads(decoded_error.decode('utf-8')) != {'error': 'unauthorized'}:
            raise RuntimeError('Encrypted response verification failed.')
        api_verified = True
        break
    except RuntimeError:
        tunnel.terminate()
        raise
    except Exception as exc:
        last_verification_error = exc
        time.sleep(1)

if not api_verified:
    tunnel.terminate()
    raise RuntimeError('Tunnel health/auth checks did not pass. Do not configure this URL.') from last_verification_error

print('Tunnel/API checks passed: WireGuard route, ML-KEM-768 key pin, AES-256-GCM request/response, and auth rejection.')
print('COLAB_RSI_ENDPOINT=' + COLAB_RSI_ENDPOINT)
print('COLAB_RSI_TOKEN=' + API_TOKEN)
print('COLAB_RSI_PQ_PUBLIC_KEY_SHA256=' + remote_fingerprint)
print('Model sync URLs:', COLAB_MODEL_URL, COLAB_MODEL_HASH_URL)
print('Keep this Colab runtime running. Copy the endpoint and token to the local client; never publish this output.')